# Mesh validation report — 2D conformal

After `femesh_gmsh`:

- `validation_report` — counts, missing grains, clockwise (should be 0 after winding unify), degenerate
- `fidelity_report` — mesh vs Shapely grain area and GB length (relative error)
- `quality_report` — aspect-ratio and min-angle stats (corners only on quadratic)

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.
Canonical mesh-only demo: `confMesh2d_gmsh.ipynb`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import box

from upxo.meshing.gsmesh2d import mesh_gs
from upxo.meshing.writer_ABQ import summarize_inp


In [ ]:
cells = {
    1: box(0, 0, 2, 2),
    2: box(2, 0, 4, 2),
    3: box(0, 2, 2, 4),
    4: box(2, 2, 4, 4),
}

In [ ]:
result = mesh_gs(cells, mesh_size_gb=0.35, mesh_size_bulk=0.7,
                 mesh_algo=6, recombine_to_quads=False)
m = result['mesher']
m.form_elsets_gmsh(); m.build_boundary_nsets(); m.build_gb_nset()
rep = m.validation_report
print('validation', rep)
print('fidelity max area err', m.fidelity_report['max_area_rel_error'])
print('fidelity max GB err', m.fidelity_report['max_gb_length_rel_error'])
print('quality', m.quality_report)
assert rep['grains_missing_elements'] == []
assert rep['degenerate_elements'] == 0
assert rep['clockwise_elements'] == 0

In [ ]:
fig, ax = m.plot_by_grain(figsize=(6, 6), show_gb=True,
                          title='validated conformal mesh')
fig

In [ ]:
out = Path.cwd() / 'confMesh2d_quality_out'
inp = m.export_abaqus_inp(out / 'rve_cps3.inp', plane='stress')
inp, summarize_inp(inp), rep